# MSCE: Hubble Tension Constraint Analysis
## All 6 Mainstream Solutions Fail Cross-Constraint Consistency Checks

**Author:** Deng Xinhang & MSCE Collaboration  
**Date:** 2026-05-30  
**DOI:** [10.5281/zenodo.20041757](https://doi.org/10.5281/zenodo.20041757)

---

### The Hubble Tension

Two independent measurements of the universe's expansion rate disagree at **~5σ**:
- **Planck 2018 (CMB):** H₀ = 67.4 ± 0.5 km/s/Mpc
- **SH0ES 2022 (Distance Ladder):** H₀ = 73.0 ± 1.0 km/s/Mpc

This notebook checks **6 mainstream resolution proposals** against **8 independent observational constraints** — simultaneously.

In [ ]:
# Cell 1: Setup
!pip install -q msce matplotlib numpy

import msce
import matplotlib.pyplot as plt
import numpy as np

print(f"MSCE version: {msce.__version__}")
print("✓ Ready")

## 1. Single-Proposal Analysis

Each of the 6 proposals is independently checked against all 8 constraints using MSCE's 6-model ensemble.

In [ ]:
# Cell 2: Load and display single-proposal results
result = msce.analyze("hubble_tension", quick=True)

# Display as table
print(f"{'Proposal':<35} {'Confidence':>10} {'Verdict':>10}")
print("-" * 57)
for key, info in result["proposals"].items():
    conf = info["confidence"]
    verdict = "✗ FAIL" if conf < 0.36 else "✓ PASS"
    print(f"{info['name']:<35} {conf:>10.3f} {verdict:>10}")

print(f"\nBest confidence: {result['confidence']:.3f}")
print(f"All proposals fail: {result['all_fail']}")

## 2. Constraint Conflict Heatmap

**Green** = pass, **Yellow** = tension (1.5-3σ), **Red** = violation.  
Each row is a constraint. Each column is a proposal. **All 6 columns have at least one red cell.**

In [ ]:
# Cell 3: Generate heatmap
fig = msce.heatmap(
    result["heatmap_data"],
    constraint_labels=result["constraint_labels"],
    proposal_labels=result["proposal_labels"],
    title="Hubble Tension: 6 Solutions × 8 Observational Constraints"
)
plt.show()

## 3. Confidence Scores

The red dashed line at 0.36 is the threshold for cross-constraint consistency.  
**All 6 proposals fall below.**

In [ ]:
# Cell 4: Confidence bar chart
proposals = [info["name"] for info in result["proposals"].values()]
scores = [info["confidence"] for info in result["proposals"].values()]

fig = msce.confidence_bars(proposals, scores, title="MSCE Confidence by Proposal")
plt.show()

## 4. Combination Search

If no single proposal passes, what about 2-factor combinations?  
Following Sam's residual direction diagnosis, we tested 4 targeted combinations.

In [ ]:
# Cell 5: Combination results
combos = result["combinations"]
print(f"{'Combination':<35} {'Confidence':>10}")
print("-" * 47)
for name, info in combos.items():
    print(f"{name:<35} {info['msce_confidence']:>10.3f}")

# Compare best single vs best combo
best_single = max(scores)
best_combo = max(c["msce_confidence"] for c in combos.values())
print(f"\nBest single proposal: {best_single:.3f}")
print(f"Best combination:     {best_combo:.3f}")
print(f"\n⚠ Combinations perform WORSE than singles — nonlinear mechanism interaction.")

## 5. Residual Direction Diagnosis

The 8D residual vector reveals **where** the proposals collectively fail.  
Highest component: **cross-constraint consistency (1.83)** → not any single observation, but the self-consistency of the ΛCDM repair framework itself.

In [ ]:
# Cell 6: Residual vector
residual = result["residual_vector"]
labels = list(residual.keys())
values = list(residual.values())

# Sort by value
sorted_idx = np.argsort(values)
labels = [labels[i] for i in sorted_idx]
values = [values[i] for i in sorted_idx]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#e74c3c" if v > 1.0 else "#f39c12" if v > 0.5 else "#2ecc71" for v in values]
ax.barh(range(len(labels)), values, color=colors, edgecolor="white")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("Average Deviation from 'All Pass'", fontsize=12)
ax.set_title("8D Constraint Residual Vector", fontsize=14, fontweight="bold")
ax.axvline(x=1.0, color="red", linestyle="--", alpha=0.5, label="High residual threshold")

for i, v in enumerate(values):
    ax.text(v + 0.03, i, f"{v:.2f}", va="center", fontsize=9)

ax.legend()
plt.tight_layout()
plt.show()

## 6. Conclusion

1. **No single-factor solution passes all 8 constraints.** Best confidence: 0.358 (DDM).
2. **2-factor combinations perform worse than singles.** Nonlinear mechanism interaction creates new conflicts.
3. **The residual direction points to the ΛCDM framework itself** — not any specific observation.
4. **The solution to the Hubble tension may require a new constraint framework** beyond ΛCDM parameter extensions.

---

### What This Means

MSCE is not saying "EDE is wrong" or "Modified Gravity is wrong."  
It's saying: **no single mechanism currently proposed can simultaneously satisfy all known observational constraints.**

The field may need to look beyond patching ΛCDM.

---

### References

- Planck Collaboration (2018). A&A 641, A6
- Riess et al. (2022). ApJL 934, L7
- DESI Collaboration (2024). arXiv:2404.03002
- Poulin et al. (2019). PRD 100, 043538 (EDE)
- Scolnic et al. (2022). ApJ 938, 113 (Pantheon+)

### Citation

```bibtex
@software{msce2026,
  title={MSCE: Multi-Source Constraint Engine},
  author={Deng, Xinhang and MSCE Collaboration},
  year={2026},
  doi={10.5281/zenodo.20041757},
  url={https://github.com/msce-ai/msce}
}
```